In [0]:
%run ./utility/logger

In [0]:
dbutils.widgets.text('catalog',"")
dbutils.widgets.text('schema',"")
dbutils.widgets.text('env',"")

In [0]:
catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')
env = dbutils.widgets.get('env')

In [0]:
print(schema)

In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW returns_incremental AS

SELECT *
FROM commerce_stage_{env}.silver.return_stage

WHERE updated_ts >
(
    SELECT COALESCE(MAX(last_processed_ts), TIMESTAMP('1900-01-01'))
    FROM commerce_main_{env}.util.etl_control
    WHERE table_name = 'fact_returns'
)
""")

In [0]:
spark.sql(f"""
INSERT INTO {catalog}.{schema}.fact_returns
(
    return_id,
    product_key,
    date_key,
    refund_amount,
    created_ts
)

SELECT

    r.return_id,

    dp.product_key,

    dd.date_key,

    r.refund_amount,

    current_timestamp()

FROM returns_incremental r

INNER JOIN {catalog}.{schema}.dim_product dp
    ON r.product_id = dp.product_id
    AND dp.is_current = true

INNER JOIN {catalog}.{schema}.dim_date dd
    ON r.return_date = dd.full_date
""")

In [0]:
spark.sql(f"""
MERGE INTO commerce_main_{env}.util.etl_control tgt

USING
(
    SELECT
        'fact_returns' AS table_name,
        MAX(updated_ts) AS last_processed_ts
    FROM commerce_stage_{env}.silver.return_stage
) src

ON tgt.table_name = src.table_name

WHEN MATCHED THEN
UPDATE SET
    tgt.last_processed_ts = src.last_processed_ts

WHEN NOT MATCHED THEN
INSERT
(
    table_name,
    last_processed_ts
)
VALUES
(
    src.table_name,
    src.last_processed_ts
)
""")